> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 04 · Strands Agents and tools

**CampusMate v1 (local agent with real campus data)**  ·  ⏱️ Time budget: **20 min theory + 60 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

In notebook 03 you gave an agent one toy tool. Now CampusMate gets **real** tools that read the (fictional) course list and the campus rulebook, so it stops guessing and starts answering from facts.

### What you'll do
- Understand the agent loop: reason → act → observe
- Build `get_course_info` and `get_campus_policy` over `data/*.json`
- Test CampusMate v1 in a loop of questions

## Settings

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib

REGION = "us-east-1"                                 # the AWS region we use in this workshop
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite (CampusMate's brain)
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # same model for quick experiments

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"                             # the fictional campus data
print("Region:", REGION, "| Model:", MODEL_ID)

## The agent loop

1. **Reason**: the model reads the question and the tool descriptions and decides what to do.
2. **Act**: it requests a tool call (toolUse).
3. **Observe**: Strands runs your Python function and feeds the result back (toolResult).
4. Repeat until the model can answer.

🏫 **Analogy:** a student solving a lab exercise: think, try a command, read the output, think again.

We test with the cheap model while we build:

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

model = BedrockModel(model_id=CHEAP_MODEL_ID, max_tokens=1000)

## Tool 1: course information

Type hints + docstring are required. The `Args:` section describes each parameter to the model.

In [ ]:
@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text())["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})

print(get_course_info("docker")[:200])   # tools are still normal functions: test them directly first

## Second tool: the rulebook

In [ ]:
@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text())["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))

print(get_campus_policy("attendance"))

## CampusMate v1

Now give both tools to an agent and ask a question that needs them.

In [ ]:
campusmate = Agent(
    model=model,
    tools=[get_course_info, get_campus_policy],
    system_prompt="You are CampusMate for AIDD students. Use tools for courses and rules; never invent rules. Be brief.",
    callback_handler=None,
)
r = campusmate("How is the Docker course assessed, and what attendance do I need to write its exam?")
print(r)
print("\nTools used:", list(r.metrics.tool_metrics))

## Test in a loop

Agents are non-deterministic, so test with several questions, including one that **should not** use a tool.

In [ ]:
questions = [
    "Which semester is Infrastructure as Code in?",
    "What is the library fine per day?",
    "Can I take part in an external hackathon during the semester?",
    "Tell me a fun fact about Bengaluru weather.",          # no tool needed
]
for q in questions:
    before = {k: v.call_count for k, v in campusmate.event_loop_metrics.tool_metrics.items()}   # counts, not live objects
    ans = campusmate(q)
    used = [k for k, v in campusmate.event_loop_metrics.tool_metrics.items() if v.call_count > before.get(k, 0)]
    print(f"Q: {q}\n   tools={used}\n   A: {str(ans).strip()[:180]}\n")

### The reference version

`projects/campusmate_agent.py` has sturdier versions of these two tools (aliases like 'exam' → semester_end_exams, semester search).
From notebook 05 on, we deploy that file. Open it now and find `get_course_info`.

## ✍️ Your turn

Write a third tool `get_event_info(keyword)` that searches `data/events.json` and returns matching events with seats left (`registration_cap - registered`). Add it to a new agent and ask about hackathons.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
@tool
def get_event_info(keyword: str) -> str:
    """TODO: describe the tool for the model (what it returns, when to use it).

    Args:
        keyword: TODO
    """
    events = json.loads((DATA_DIR / "events.json").read_text())["events"]
    # TODO: filter events whose title/type/description contains keyword, add "seats_left"
    return json.dumps({"todo": True})

print(get_event_info("hackathon"))
# TODO: agent = Agent(model=model, tools=[get_course_info, get_campus_policy, get_event_info], callback_handler=None)
# TODO: print(agent("Is there any hackathon with seats left?"))

## ✅ Checkpoint

You are on track if:
- The v1 answer mentions the AIDD301 assessment split (30/20/50) and 75% attendance
- 'Tools used' lists both tools; the Bengaluru question used none
- Your `get_event_info` returns EVT-03 and EVT-06 for 'hackathon'

**Stuck or behind?** Copy from `projects/campusmate_agent.py (tools section)` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'strands_agents'` | Wrong import from old tutorials | Use `from strands import Agent, tool` |
| Tool never called | Vague docstring, or the system prompt doesn't mention tools | Make the docstring say *when* to use it |
| `AccessDeniedException ... model access` | Nova 2 Lite isn't enabled in your account | Bedrock console → **Model access** → enable Amazon Nova 2 Lite |
| `FileNotFoundError ... data/courses.json` | Kernel started outside the repo | Restart Jupyter from the repo root |